In [0]:
import pandas as pd
from pyspark.sql import functions as F

CAMADAS = {
    "bronze": "/Volumes/workspace/olist_bronze/parquet",   # ajuste
    "silver": "/Volumes/workspace/olist_silver/parquet",   # ajuste
    "gold":   "/Volumes/workspace/olist_gold/parquet",
    "ml":     "/Volumes/workspace/olist_gold/ml_predictions",
}
PALAVRAS_CHAVE = ("id", "category", "uf_", "mes", "data", "zip", "state")

linhas = []
for camada, base in CAMADAS.items():
    try:
        pastas = [p for p in dbutils.fs.ls(base) if not p.name.startswith(("_", "."))]
    except Exception as e:
        print(f"{camada}: caminho não encontrado ({base})"); continue
    for p in pastas:
        tabela = p.name.rstrip("/").replace(".parquet", "")
        try:
            df = spark.read.parquet(p.path)
        except Exception:
            continue
        n = df.count()
        particoes = [s.name.split("=")[0] for s in dbutils.fs.ls(p.path) if "=" in s.name] if p.isDir() else []
        for c, t in df.dtypes:
            info = {"camada": camada, "tabela": tabela, "linhas": n, "coluna": c, "tipo": t,
                    "particao": c in particoes,
                    "pct_nulos": round(df.filter(F.col(c).isNull()).count() / n * 100, 2) if n else None,
                    "distintos": None}
            if any(k in c.lower() for k in PALAVRAS_CHAVE):
                info["distintos"] = df.select(c).distinct().count()
            linhas.append(info)

resultado = pd.DataFrame(linhas)
display(resultado)
resultado.to_csv("/Volumes/workspace/olist_gold/parquet/_estrutura_tabelas.csv", index=False)